In [54]:
import pinecone
from pinecone import Pinecone, ServerlessSpec
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer

In [55]:
load_dotenv(find_dotenv(), override=True)

True

In [56]:
files = pd.read_csv("course_description_1.csv", encoding="ANSI")

In [57]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]} the slug is {row["course_slug"]}, the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

In [58]:
files['course_description_new'] = files.apply(create_course_description, axis=1)
print(files['course_description_new'] )

0      The course name is Introduction to Tableau the...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
                             ...                        
101    The course name is Intro to NLP for AI the slu...
102    The course name is Data Analysis with ChatGPT ...
103    The course name is ChatGPT for Data Science th...
104    The course name is Intro to LLMs the slug is i...
105    The course name is Growth Analysis with SQL, P...
Name: course_description_new, Length: 106, dtype: object


In [59]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"), endpoint=os.getenv("PINECONE_ENV"))

In [60]:
index_name = "my-index"
dimension = 768
metric = "cosine"

In [61]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")

my-index succesfully deleted.


In [62]:
pc.create_index(
    name = index_name,
    dimension = dimension,
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1")
    )

{
    "name": "my-index",
    "metric": "cosine",
    "host": "my-index-5blbnge.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null
}

In [63]:
index = pc.Index(index_name)

In [64]:
model = SentenceTransformer('multi-qa-distilbert-cos-v1')

In [65]:
def create_embeddings(row):
    combined_text = ' '.join([str(row[field]) for field in ['course_description', 'course_description_new', 'course_description_short']])
    embedding = model.encode(combined_text, show_progress_bar = False)
    return embedding

In [66]:
files["embedding"] = files.apply(create_embeddings, axis = 1)

In [67]:
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]

In [68]:
index.upsert(vectors = vectors_to_upsert)

{'upserted_count': 106}

In [69]:
index.upsert(vectors = vectors_to_upsert)

{'upserted_count': 106}

In [82]:
query = "clustering"

In [83]:
query_embedding = model.encode(query, show_progress_bar = False).tolist()

In [92]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12,
)

In [93]:
query_results

{'matches': [{'id': 'Machine Learning in Excel',
              'score': 0.295622855,
              'values': []},
             {'id': 'Machine Learning with K-Nearest Neighbors',
              'score': 0.241912872,
              'values': []},
             {'id': 'Machine Learning in Python',
              'score': 0.19538404,
              'values': []},
             {'id': 'Growth Analysis with SQL, Python, and Tableau  ',
              'score': 0.180904418,
              'values': []},
             {'id': 'The Complete Data Visualization Course with Python, R, '
                    'Tableau, and Excel',
              'score': 0.161060363,
              'values': []},
             {'id': 'Introduction to R Programming',
              'score': 0.158472553,
              'values': []},
             {'id': 'Mathematics', 'score': 0.147547737, 'values': []},
             {'id': 'Excel for Project Management',
              'score': 0.147285476,
              'values': []},
             {